# 02_Data_Cleaning

**Project**: Analysis of Socioeconomic Inequality in the ENEM 2023

**Author**: Luís Antonio Manfredi Sodré

**Date**: 2026-24-06

## Purpose

Loads the raw microdata, filters to participants who attended both exam days
and received a valid essay score, then maps coded values to descriptive labels.

## 1. Load & Filter Raw Data

In [ ]:
import duckdb
import pandas as pd

In [ ]:
df = duckdb.sql("""
    SELECT
        TP_SEXO,
        TP_COR_RACA,
        TP_ESCOLA,
        SG_UF_PROVA,
        NU_NOTA_CN,
        NU_NOTA_CH,
        NU_NOTA_LC,
        NU_NOTA_MT,
        NU_NOTA_REDACAO,
        Q006,
        Q022,
        Q024,
        Q025
    FROM read_csv(
    '../data/raw/MICRODADOS_ENEM_2023.csv',
    delim=';',
    encoding='latin-1'
    )
    WHERE TP_PRESENCA_CN = 1
        AND TP_PRESENCA_CH = 1
        AND TP_PRESENCA_LC = 1
        AND TP_PRESENCA_MT = 1
        AND TP_STATUS_REDACAO = 1
""").df()

## 2. Map Coded Values

In [ ]:
df["TP_SEXO"] = df["TP_SEXO"].replace({
    "F": "Female",
    "M": "Male"
})

In [ ]:
df["TP_COR_RACA"] = df["TP_COR_RACA"].replace({
    0: "Not Informed",
    1: "White",
    2: "Black",
    3: "Brown",
    4: "Asian",
    5: "Indigenous"
})

In [ ]:
df["TP_ESCOLA"] = df["TP_ESCOLA"].replace({
    1: "Didn't respond",
    2: "Public",
    3: "Private"
})

In [ ]:
df["Q006"] = df["Q006"].replace({
    "A": "No Income",
    "B": "Up to 1.320,00",
    "C": "1.320,01-1.980,00",
    "D": "1.980,01-2.640,00",
    "E": "2.640,01-3.300,00",
    "F": "3.300,01-3.960,00",
    "G": "3.960,01-5.280,00",
    "H": "5.280,01-6.600,00",
    "I": "6.600,01-7.920,00",
    "J": "7.920,01-9.240,00",
    "K": "9.240,01-10.560,00",
    "L": "10.560,01-11.880,00",
    "M": "11.880,01-13.200,00",
    "N": "13.200,01-15.840,00",
    "O": "15.840,01-19.800,00",
    "P": "19.800,01-26.400,00",
    "Q": "Above 26.400,00"
})

In [ ]:
df["Q022"] = df["Q022"].replace({
    "A": "0",
    "B": "1",
    "C": "2",
    "D": "3",
    "E": "4+"
})

In [ ]:
df["Q024"] = df["Q024"].replace({
    "A": "0",
    "B": "1",
    "C": "2",
    "D": "3",
    "E": "4+"
})

In [ ]:
df["Q025"] = df["Q025"].replace({
    "A": "No",
    "B": "Yes"
})

## 3. Apply Ordered Categories

In [ ]:
income_order = [
    "No Income", "Up to 1.320,00", "1.320,01-1.980,00",
    "1.980,01-2.640,00", "2.640,01-3.300,00", "3.300,01-3.960,00",
    "3.960,01-5.280,00", "5.280,01-6.600,00", "6.600,01-7.920,00",
    "7.920,01-9.240,00", "9.240,01-10.560,00", "10.560,01-11.880,00",
    "11.880,01-13.200,00", "13.200,01-15.840,00", "15.840,01-19.800,00",
    "19.800,01-26.400,00", "Above 26.400,00"
]

df["Q006"] = pd.Categorical(
    df["Q006"], 
    categories=income_order, 
    ordered=True)

In [ ]:
device_order = ["0", "1", "2", "3", "4+"]

df["Q022"] = pd.Categorical(
    df["Q022"],
    categories=device_order,
    ordered=True
)

df["Q024"] = pd.Categorical(
    df["Q024"],
    categories=device_order,
    ordered=True
)

## 4. Validate & Export

In [ ]:
print(df.info())
print("\nNaN por coluna:")
print(df.isna().sum())

In [ ]:
output_path = "../data/processed/enem_2023_clean.parquet"
df.to_parquet(output_path, index=False)
print(f"Saved in: {output_path}")
print(f"Shape: {df.shape}")